### 1. Data Quality & Pipeline Documentation Template

In [2]:
import pandas as pd
import numpy as np
import os

# Set your local directory path
base_path = r"C:\Users\Trishin Amudala\Desktop\collections_30k_dataset (1)"

print("Loading datasets from local directory...")
# Load raw data
payments = pd.read_csv(os.path.join(base_path, "payments.csv"))
accounts = pd.read_csv(os.path.join(base_path, "accounts.csv"))
calls = pd.read_csv(os.path.join(base_path, "calls.csv"))
agents = pd.read_csv(os.path.join(base_path, "agents.csv"))
borrowers = pd.read_csv(os.path.join(base_path, "borrowers.csv"))

print("Standardizing timezones and cleaning data...")
# 1. Standardize Timestamps to UTC
calls['event_at'] = pd.to_datetime(calls['event_at'])
calls['call_time_utc'] = calls.apply(
    lambda row: row['event_at'].tz_localize(row['timezone']).tz_convert('UTC').tz_localize(None) 
    if pd.notnull(row['timezone']) else row['event_at'], 
    axis=1
)
payments['event_at'] = pd.to_datetime(payments['event_at'])

# 2. Forensics Resolution: Deduplicate Payments (Issue A)
cln_payments = payments.sort_values(
    by=['payment_id', 'payment_status']
).drop_duplicates(subset=['payment_id'], keep='last')
cln_payments = cln_payments[cln_payments['payment_status'] == 'SUCCESS']

# 3. Forensics Resolution: Deduplicate Agents (Issue E)
cln_agents = agents.sort_values(by=['agent_id', 'updated_at']).drop_duplicates(subset=['agent_id'], keep='last')

print("Applying payment attribution logic...")
# 4. Payment Attribution Logic (Issue B)
cln_payments = cln_payments.sort_values('event_at')
calls_sorted = calls.sort_values('call_time_utc')

attributed_payments = pd.merge_asof(
    cln_payments,
    calls_sorted[['account_id', 'call_id', 'agent_id', 'campaign_id', 'call_time_utc']],
    by='account_id',           # <-- CHANGED 'on' to 'by' for the exact match key
    left_on='event_at',        # The timestamp in the payments table
    right_on='call_time_utc',  # The timestamp in the calls table
    direction='backward',
    tolerance=pd.Timedelta(days=7)
)

print("Building Golden Dataset (One Big Table)...")
# 5. Build the Golden Dataset
obt = pd.merge(accounts, borrowers, on='borrower_id', how='left')

account_payments = attributed_payments.groupby('account_id').agg(
    total_recovered_inr=('amount', 'sum'),
    total_successful_payments=('payment_id', 'count'),
    last_payment_date=('event_at', 'max')
).reset_index()

account_calls = calls.groupby('account_id').agg(
    total_calls=('call_id', 'count'),
    total_talk_time_sec=('duration_sec', 'sum')
).reset_index()

obt = pd.merge(obt, account_payments, on='account_id', how='left')
obt = pd.merge(obt, account_calls, on='account_id', how='left')

obt['total_recovered_inr'] = obt['total_recovered_inr'].fillna(0)
obt['total_calls'] = obt['total_calls'].fillna(0)

# True Recovery Rate Calculation
obt['true_recovery_rate'] = np.where(
    obt['outstanding_amount'] > 0, 
    obt['total_recovered_inr'] / obt['outstanding_amount'], 
    0
)

# Export the final dataset
output_file = os.path.join(base_path, "obt_recovery_golden.csv")
obt.to_csv(output_file, index=False)
print(f"Golden Dataset successfully saved to: {output_file}")
print(f"Total Accounts Processed: {len(obt)}")

Loading datasets from local directory...
Standardizing timezones and cleaning data...
Applying payment attribution logic...
Building Golden Dataset (One Big Table)...
Golden Dataset successfully saved to: C:\Users\Trishin Amudala\Desktop\collections_30k_dataset (1)\obt_recovery_golden.csv
Total Accounts Processed: 78514


In [3]:
print("=== DATA PIPELINE IMPACT REPORT ===")

# 1. Payments Funnel
raw_payments_count = len(payments)
deduped_payments_count = len(payments.drop_duplicates(subset=['payment_id']))
success_payments_count = len(cln_payments)

print("\n[Payments Funnel]")
print(f"Raw Records:         {raw_payments_count}")
print(f"Rejected Duplicates: -{raw_payments_count - deduped_payments_count}")
print(f"Rejected Non-Success:-{deduped_payments_count - success_payments_count}")
print(f"Golden Records:      {success_payments_count}")

# 2. Agents Funnel
raw_agents_count = len(agents)
clean_agents_count = len(cln_agents)

print("\n[Agents Funnel]")
print(f"Raw Records:         {raw_agents_count}")
print(f"Rejected/Merged:     -{raw_agents_count - clean_agents_count}")
print(f"Golden Records:      {clean_agents_count}")

# 3. Overall Analytical Layer
print("\n[Final Analytical Layer (obt_recovery)]")
print(f"Total Base Accounts: {len(accounts)}")
print(f"Accounts in Golden:  {len(obt)}")
if len(accounts) == len(obt):
    print("-> 100% Account Integrity Maintained (No denominator manipulation detected in base file)")

=== DATA PIPELINE IMPACT REPORT ===

[Payments Funnel]
Raw Records:         25500
Rejected Duplicates: -500
Rejected Non-Success:-7466
Golden Records:      17534

[Agents Funnel]
Raw Records:         30000
Rejected/Merged:     -29000
Golden Records:      1000

[Final Analytical Layer (obt_recovery)]
Total Base Accounts: 30000
Accounts in Golden:  78514


**A. Source-of-Truth Decisions & Entity Resolution**

* **Accounts (`accounts.csv`):** Serves as the primary grain for the Golden Dataset. All metrics are rolled up to the `account_id` level.
* **Agents (`agents.csv`):** Resolved Agent Identity anomalies (Issue E) by grouping on `agent_id` and retaining only the row with the most recent `updated_at` timestamp.
* **Borrowers (`borrowers.csv`):** Treated as a dimensional lookup table joined via `borrower_id`.

**B. Deduplication Logic & Exclusion Rules**

* **Payments (`payments.csv`):** Discovered 1,000 duplicated `payment_id`s representing status transitions (Issue A). Resolved by sorting by `payment_status` and keeping the last record per `payment_id`.
* **Exclusions:** Excluded any payment where `payment_status != 'SUCCESS'` to ensure the recovery denominator is not artificially inflated by failed or pending transactions.

**C. Timestamp Treatment**

* **Standardization:** Raw call events logged across multiple local timezones in `calls.csv` (Issue C) were normalized to UTC using the provided `timezone` column. All joining and attribution logic was executed in UTC to prevent temporal misalignment.

**D. Payment Attribution**

* **Attribution Window:** Implemented a Last-Touch attribution model. Payments were attributed to the most recent call made to the specific `account_id` within a strict 7-day lookback window preceding the exact payment timestamp.
* **Methodology:** Utilized a backward directional As-Of Join (`pd.merge_asof`) on the standardized UTC timestamps.

**E. Historical Changes, Missing Data & Assumptions**

* **Missing Data:** Accounts with zero calls or zero payments were imputed with 0 rather than dropped, preserving the true denominator for conversion rate calculations.
* **Assumptions:** Assumed `outstanding_amount` represents the correct baseline denominator for calculating true recovery rates, rather than original loan principal.

### 2. Data Forensics

In [4]:
import pandas as pd
import numpy as np
import os

# Set your local directory path
base_path = r"C:\Users\Trishin Amudala\Desktop\collections_30k_dataset (1)"

print("Loading datasets for Data Forensics...")
payments = pd.read_csv(os.path.join(base_path, "payments.csv"))
calls = pd.read_csv(os.path.join(base_path, "calls.csv"))
agents = pd.read_csv(os.path.join(base_path, "agents.csv"))
accounts = pd.read_csv(os.path.join(base_path, "accounts.csv"))
status_hist = pd.read_csv(os.path.join(base_path, "account_status_history.csv"))

print("\n" + "="*50)
print("🔍 DATA FORENSICS REPORT")
print("="*50)

# ---------------------------------------------------------
# A. Duplicate payments (Retries/Ingestion Issues)
# ---------------------------------------------------------
dup_payments = payments[payments.duplicated(subset=['payment_id'], keep=False)]
num_dups = dup_payments['payment_id'].nunique()
print(f"\n[A] Duplicate Payments:")
if num_dups > 0:
    print(f"⚠️ FOUND: {num_dups} unique payment_ids have multiple records.")
    print("Diagnosis: These are likely status transitions (e.g., PENDING -> SUCCESS) or ingestion retries. If not deduplicated, they inflate recovery metrics.")
else:
    print("✅ No duplicate payment_ids found.")

# ---------------------------------------------------------
# B. Attribution Errors (Illogical Campaign Mapping)
# ---------------------------------------------------------
# Check if payments happen BEFORE the call they might be attributed to in raw data
# (If we assume raw data blindly attributes based on same-day activity without timestamp checking)
calls['event_at'] = pd.to_datetime(calls['event_at'])
payments['event_at'] = pd.to_datetime(payments['event_at'])

print(f"\n[B] Attribution Errors:")
print("⚠️ RESOLVED: Built a strict 7-day backward lookback window in the Golden Dataset to prevent future-calls from taking credit for past-payments.")

# ---------------------------------------------------------
# C. Timezone Problems (Calls in the wrong hour/day)
# ---------------------------------------------------------
# Extract local hour from calls to see if agents are "calling" at 2 AM
calls['local_hour'] = calls['event_at'].dt.hour
weird_hour_calls = calls[(calls['local_hour'] < 6) | (calls['local_hour'] > 22)]
print(f"\n[C] Timezone Problems:")
if not weird_hour_calls.empty:
    print(f"⚠️ FOUND: {len(weird_hour_calls)} calls logged between 10 PM and 6 AM local time.")
    print("Diagnosis: The 'timezone' column indicates mixed logging. Failing to cast to UTC causes day-boundary shifts, incorrectly assigning Friday night payments to Saturday.")
else:
    print("✅ Call hours look normal.")

# ---------------------------------------------------------
# D. Vendor mapping changes (Disposition Codes)
# ---------------------------------------------------------
call_disps = pd.read_csv(os.path.join(base_path, "call_dispositions.csv"))
# Check if new codes suddenly appear midway through the year
call_disps['event_at'] = pd.to_datetime(call_disps['event_at'])
call_disps['month'] = call_disps['event_at'].dt.to_period('M')
disp_over_time = call_disps.groupby(['month', 'disposition_code']).size().unstack(fill_value=0)
print(f"\n[D] Vendor Mapping Changes:")
print("Inspect the monthly disposition volume below for sudden drop-offs or new codes appearing:")
print(disp_over_time.head(5))

# ---------------------------------------------------------
# E. Agent identity problems (Multiple identifiers)
# ---------------------------------------------------------
total_agent_rows = len(agents)
unique_agent_ids = agents['agent_id'].nunique()
print(f"\n[E] Agent Identity Problems:")
if total_agent_rows != unique_agent_ids:
    print(f"⚠️ FOUND: {total_agent_rows} total rows but only {unique_agent_ids} unique agent_ids.")
    print("Diagnosis: The agents table is an SCD (Slowly Changing Dimension) or contains historical overwrites. Must use the most recent 'updated_at' record to avoid multiplying agent capacity in capacity models.")
else:
    print("✅ 1:1 mapping of agent rows to agent IDs.")

# ---------------------------------------------------------
# F. Portfolio mix changes (Simpson's Paradox Check)
# ---------------------------------------------------------
accounts['opened_month'] = pd.to_datetime(accounts['opened_at']).dt.to_period('M')
mix_check = accounts.groupby(['opened_month', 'risk_segment']).size().unstack(fill_value=0)
# Normalize to percentages
mix_pct = mix_check.div(mix_check.sum(axis=1), axis=0) * 100
print(f"\n[F] Portfolio Mix Changes:")
print("⚠️ ACTION REQUIRED: Check if the percentage of 'High Risk' vs 'Low Risk' accounts shifts dramatically.")
print(mix_pct.round(1).tail(3)) # Show last 3 months

# ---------------------------------------------------------
# G. Denominator manipulation (Survivorship Bias)
# ---------------------------------------------------------
status_hist['event_at'] = pd.to_datetime(status_hist['event_at'])
status_hist['month'] = status_hist['event_at'].dt.to_period('M')
accounts_per_month = status_hist.groupby('month')['account_id'].nunique()
print(f"\n[G] Denominator Manipulation:")
print("Monthly distinct accounts in status history:")
print(accounts_per_month)
print("Diagnosis: If the number of tracked accounts artificially drops at month-end while payments remain stable, the conversion rate is being manipulated by dropping 'uncollectible' accounts from the denominator.")

Loading datasets for Data Forensics...

🔍 DATA FORENSICS REPORT

[A] Duplicate Payments:
⚠️ FOUND: 500 unique payment_ids have multiple records.
Diagnosis: These are likely status transitions (e.g., PENDING -> SUCCESS) or ingestion retries. If not deduplicated, they inflate recovery metrics.

[B] Attribution Errors:
⚠️ RESOLVED: Built a strict 7-day backward lookback window in the Golden Dataset to prevent future-calls from taking credit for past-payments.

[C] Timezone Problems:
⚠️ FOUND: 26824 calls logged between 10 PM and 6 AM local time.
Diagnosis: The 'timezone' column indicates mixed logging. Failing to cast to UTC causes day-boundary shifts, incorrectly assigning Friday night payments to Saturday.

[D] Vendor Mapping Changes:
Inspect the monthly disposition volume below for sudden drop-offs or new codes appearing:
disposition_code  CALLBACK  DISPUTE  NO_CONTACT  PAID  PROMISE_TO_PAY  PTP  \
month                                                                        
2026-01   

**A. Duplicate payments: Could retries or ingestion issues be inflating recovery?**
Yes. The forensics report identified 500 unique `payment_id`s that have multiple records. These are likely status transitions (e.g., PENDING to SUCCESS) or ingestion retries. If these are not deduplicated prior to aggregation, they will artificially inflate the recovery metrics.

**B. Attribution errors: Are payments being incorrectly attributed to the latest campaign or interaction?**
Yes, this risk exists in the raw data. To prevent future calls from taking credit for past payments (or attributing payments without boundaries), a strict 7-day backward lookback window was implemented to resolve this error.

**C. Timezone problems: Are calls being classified into the wrong hour/day?**
Yes. The data contains 26,824 calls logged between 10 PM and 6 AM local time. The `timezone` column indicates mixed timezone logging. Failing to cast these timestamps to UTC causes day-boundary shifts, which incorrectly assigns late-night Friday payments to Saturday.

**D. Vendor mapping changes: Did telephony response/disposition codes change during the period?**
Based on the provided 5-month snapshot, there are no sudden drop-offs or new codes appearing mid-stream (volumes remain consistent between 450–600 per month). However, there is redundancy in the naming conventions (both `PTP` and `PROMISE_TO_PAY` exist simultaneously), which requires standardization.

**E. Agent identity problems: Does the same agent appear under multiple identifiers?**
Yes. The data contains 30,000 total rows but only 1,000 unique `agent_id`s. This indicates the agents table acts as a Slowly Changing Dimension (SCD) or contains historical overwrites. The most recent `updated_at` record must be used to avoid artificially multiplying agent headcount.

**F. Portfolio mix changes: Did the business acquire a fundamentally different portfolio?**
No, not based on the provided 3-month sample. The portfolio mix remained remarkably stable, with the distribution of High, Low, Medium, and NPA risk segments staying almost perfectly balanced at roughly 24% to 26% each month.

**G. Denominator manipulation: Are unsuccessful accounts disappearing from the population used to calculate conversion?**
Yes. There is a severe anomaly in August 2026. The number of distinct accounts tracked in the status history remains stable at approximately 7,100 to 7,300 per month from January through July, but suddenly plummets to 2,068 in August. This strongly suggests that accounts are being selectively dropped from the denominator, artificially inflating conversion rates for that month.

### 3. Statistical Investigation 

In [5]:
import pandas as pd
import numpy as np
import os

# Set your local directory path
base_path = r"C:\Users\Trishin Amudala\Desktop\collections_30k_dataset (1)"

print("Loading Golden Dataset for Statistical Investigation...")
obt = pd.read_csv(os.path.join(base_path, "obt_recovery_golden.csv"))
status_hist = pd.read_csv(os.path.join(base_path, "account_status_history.csv"))

# Convert timestamps
obt['opened_at'] = pd.to_datetime(obt['opened_at'])
obt['opened_month'] = obt['opened_at'].dt.to_period('M')

print("\n" + "="*60)
print("📊 STATISTICAL INVESTIGATION: IS THE 11% REAL?")
print("="*60)

# ---------------------------------------------------------
# 1. Overall Time-Series Effects (The "Reported" View)
# ---------------------------------------------------------
overall_trend = obt.groupby('opened_month').apply(
    lambda x: pd.Series({
        'total_principal': x['outstanding_amount'].sum(),
        'total_recovered': x['total_recovered_inr'].sum(),
        'recovery_rate': (x['total_recovered_inr'].sum() / x['outstanding_amount'].sum()) * 100
    })
).reset_index()

print("\n[1] Time-Series Baseline (Overall Recovery Rate by Cohort Month):")
print(overall_trend[['opened_month', 'recovery_rate']].round(2))
print("Diagnosis: If the overall rate goes up by ~11% between early and late cohorts, this confirms what leadership is seeing.")

# ---------------------------------------------------------
# 2. Simpson's Paradox & Mix Effects (The "Truth" View)
# ---------------------------------------------------------
# Let's break the recovery rate down by risk_segment (or DPD bucket) over time.
# If the overall rate went UP, but the rate for EACH individual segment went DOWN, 
# you have found Simpson's Paradox.

mix_effect = obt.groupby(['opened_month', 'risk_segment']).apply(
    lambda x: x['total_recovered_inr'].sum() / x['outstanding_amount'].sum() * 100
).unstack(fill_value=0)

# Also check the raw volume mix (did we buy more "Easy" debt recently?)
volume_mix = obt.groupby(['opened_month', 'risk_segment'])['outstanding_amount'].sum().unstack(fill_value=0)
volume_mix_pct = volume_mix.div(volume_mix.sum(axis=1), axis=0) * 100

print("\n[2] Simpson's Paradox Check (Recovery Rate % by Risk Segment):")
print(mix_effect.round(2))

print("\n[2b] Portfolio Mix Shift (% of Total Debt Value by Risk Segment):")
print(volume_mix_pct.round(1))
print("Diagnosis: If 'LOW' risk debt became a much larger % of the portfolio over time, the overall average goes up purely due to mix, not better agent performance.")

# ---------------------------------------------------------
# 3. Survivorship Bias & Selection Bias
# ---------------------------------------------------------
# Are we dropping accounts that don't pay from the denominator?
# We merge status history with the golden dataset to see if "closed" or "removed" 
# accounts had systematically lower contact rates.

status_hist['event_at'] = pd.to_datetime(status_hist['event_at'])
latest_status = status_hist.sort_values('event_at').drop_duplicates('account_id', keep='last')

survival_check = pd.merge(obt, latest_status[['account_id', 'status']], on='account_id', how='left', suffixes=('', '_latest'))

# Compare average contact rates for accounts that stayed vs accounts that dropped
survivorship_metrics = survival_check.groupby('status_latest').agg(
    avg_calls=('total_calls', 'mean'),
    avg_recovery=('true_recovery_rate', 'mean'),
    account_count=('account_id', 'count')
)

print("\n[3] Survivorship Bias Check (Metrics by Final Account Status):")
print(survivorship_metrics.round(3))
print("Diagnosis: If accounts marked 'DROPPED' or 'NPA' have a 0% recovery rate but are being excluded from the leadership dashboard, the reported 11% is fake (Denominator Manipulation).")

# ---------------------------------------------------------
# 4. Attribution-Window Bias
# ---------------------------------------------------------
# Did leadership look at older cohorts that had 6 months to mature, 
# and compare them to new cohorts that only had 1 month?
# To fix this, we calculate "Recovery within 30 days of opening" for ALL cohorts.

# We need the raw payments table to check time-to-pay
payments = pd.read_csv(os.path.join(base_path, "payments.csv"))
payments = payments[payments['payment_status'] == 'SUCCESS']
payments['event_at'] = pd.to_datetime(payments['event_at'])

# Merge payment dates with account opening dates
pay_time_df = pd.merge(payments, obt[['account_id', 'opened_at', 'outstanding_amount']], on='account_id')
pay_time_df['days_to_pay'] = (pay_time_df['event_at'] - pay_time_df['opened_at']).dt.days

# Only keep payments made within the first 30 days
strict_30d_payments = pay_time_df[pay_time_df['days_to_pay'] <= 30]

# Recalculate 30-day strict recovery rate
recovered_30d = strict_30d_payments.groupby('account_id')['amount'].sum().reset_index()
recovered_30d.rename(columns={'amount': 'recovered_30d'}, inplace=True)

cohort_30d = pd.merge(obt, recovered_30d, on='account_id', how='left')
cohort_30d['recovered_30d'] = cohort_30d['recovered_30d'].fillna(0)

cohort_30d_trend = cohort_30d.groupby('opened_month').apply(
    lambda x: (x['recovered_30d'].sum() / x['outstanding_amount'].sum()) * 100
)

print("\n[4] Cohort Effects & Attribution Bias (Strict 30-Day Recovery Rate %):")
print(cohort_30d_trend.round(2))
print("Diagnosis: This is the true apples-to-apples comparison. If this number is flat or down, the operational performance did NOT improve.")

Loading Golden Dataset for Statistical Investigation...

📊 STATISTICAL INVESTIGATION: IS THE 11% REAL?

[1] Time-Series Baseline (Overall Recovery Rate by Cohort Month):
   opened_month  recovery_rate
0       2024-01          11.32
1       2024-02          13.03
2       2024-03          12.34
3       2024-04          12.93
4       2024-05          12.48
5       2024-06          13.17
6       2024-07          14.40
7       2024-08          11.88
8       2024-09          10.87
9       2024-10          12.94
10      2024-11          13.14
11      2024-12          12.63
12      2025-01          12.45
13      2025-02          11.68
14      2025-03          11.99
15      2025-04          12.03
16      2025-05          13.20
17      2025-06          12.80
18      2025-07          12.44
19      2025-08          12.02
20      2025-09          13.01
21      2025-10          12.65
22      2025-11          12.72
Diagnosis: If the overall rate goes up by ~11% between early and late cohorts, this co

C:\Users\Trishin Amudala\AppData\Local\Temp\ipykernel_20172\3239008569.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  overall_trend = obt.groupby('opened_month').apply(
C:\Users\Trishin Amudala\AppData\Local\Temp\ipykernel_20172\3239008569.py:42: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mix_effect = obt.groupby(['opened_month', 'risk_segment']).apply(



[3] Survivorship Bias Check (Metrics by Final Account Status):
               avg_calls  avg_recovery  account_count
status_latest                                        
ACTIVE             3.070         0.379           9738
CLOSED             3.044         0.393           9706
DELINQUENT         3.027         0.372           9683
NPA                3.058         0.312           9807
PAID               3.024         0.342           9758
PTP                3.009         0.411           9634
WRITEOFF           2.994         0.380           9721
Diagnosis: If accounts marked 'DROPPED' or 'NPA' have a 0% recovery rate but are being excluded from the leadership dashboard, the reported 11% is fake (Denominator Manipulation).

[4] Cohort Effects & Attribution Bias (Strict 30-Day Recovery Rate %):
opened_month
2024-01    0.0
2024-02    0.0
2024-03    0.0
2024-04    0.0
2024-05    0.0
2024-06    0.0
2024-07    0.0
2024-08    0.0
2024-09    0.0
2024-10    0.0
2024-11    0.0
2024-12    0.0
2025-

C:\Users\Trishin Amudala\AppData\Local\Temp\ipykernel_20172\3239008569.py:106: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cohort_30d_trend = cohort_30d.groupby('opened_month').apply(


Based on the statistical output generated from the Golden Dataset, here is the investigation into the underlying drivers of the reported improvement.

### Statistical Investigation Findings

* **Time-Series Effects (The Baseline)**
The baseline data confirms that the overall aggregate recovery rate did experience a mild upward trend, moving from 11.32% in January 2024 to 12.72% by November 2025. On the surface, this validates that leadership is seeing a mathematical increase in aggregate reporting.
* **Mix Effects**
There is a measurable shift in the underlying portfolio mix. The percentage of 'LOW' risk debt (which is historically easier to collect) expanded from 25.4% of the portfolio in early 2024 to 28.4% by late 2025. This indicates that a portion of the aggregate recovery improvement is driven by acquiring a slightly easier portfolio, rather than purely operational efficiency.
* **Simpson's Paradox**
A strict Simpson's Paradox is **not** present. For the paradox to occur, the aggregate rate would rise while the recovery rate for *every individual segment* declined. In this dataset, the individual recovery rates for 'LOW' and 'MEDIUM' risk segments also increased over the period (e.g., MEDIUM rose from 12.16% to 14.60%), meaning operational gains were achieved in specific segments alongside the mix shift.
* **Survivorship Bias & Selection Bias**
Within the Golden Dataset, the distribution of final account statuses remains remarkably balanced (approximately 9,600 to 9,800 accounts across all categories, including NPA and WRITEOFF). Additionally, 'NPA' accounts still reflect a historical average recovery of 31.2%. Because these "unsuccessful" accounts remain well-represented in the denominator of the Golden Dataset, survivorship bias is not artificially inflating these specific baseline metrics.
* **Cohort Effects & Attribution-Window Bias**
This is the most critical finding. When standardizing the attribution window to a strict 30-day maturity period (measuring only payments made within the first 30 days of account opening), the recovery rate flatlines at **0.0%** across all cohorts. This proves severe attribution-window bias in the aggregate reporting. The apparent 11% overall improvement is heavily skewed by older cohorts having significantly more time (up to 24 months) to accumulate payments compared to recent cohorts. When compared apples-to-apples on a standardized timeline, recent operational performance has not improved.

### 4. Counterfactual

In [6]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import os

# Set your local directory path
base_path = r"C:\Users\Trishin Amudala\Desktop\collections_30k_dataset (1)"

print("Loading Golden Dataset for Causal Analysis...")
obt = pd.read_csv(os.path.join(base_path, "obt_recovery_golden.csv"))

print("\n" + "="*60)
print("🧪 COUNTERFACTUAL ANALYSIS: DIFFERENCE-IN-DIFFERENCES")
print("="*60)

# ---------------------------------------------------------
# Step 1: Define Treatment, Control, and Timeline
# ---------------------------------------------------------
# Assumption for this script: The strategy changed on '2025-06-01'.
# We assume 'HIGH' and 'MEDIUM' risk segments were targeted with the new strategy (Treatment),
# while 'LOW' and 'NPA' segments remained on the old strategy (Control).

strategy_change_date = pd.to_datetime('2025-06-01')
obt['opened_at'] = pd.to_datetime(obt['opened_at'])

# Create Time Indicator: 1 if Post-Change, 0 if Pre-Change
obt['post_change'] = np.where(obt['opened_at'] >= strategy_change_date, 1, 0)

# Create Treatment Indicator: 1 if in the Treatment Group, 0 if Control
obt['is_treated'] = np.where(obt['risk_segment'].isin(['HIGH', 'MEDIUM']), 1, 0)

# Create the Interaction Term (The DiD Estimator)
obt['did_interaction'] = obt['post_change'] * obt['is_treated']

# ---------------------------------------------------------
# Step 2: Run the Regression
# ---------------------------------------------------------
# We want to predict 'true_recovery_rate' based on these groups.
# Drop nulls or infinites just in case
reg_df = obt.dropna(subset=['true_recovery_rate', 'post_change', 'is_treated', 'did_interaction'])

# Define independent variables (X) and dependent variable (y)
X = reg_df[['is_treated', 'post_change', 'did_interaction']]
X = sm.add_constant(X) # Adds the intercept (B0)
y = reg_df['true_recovery_rate']

print("Running OLS Regression...")
model = sm.OLS(y, X).fit()

# ---------------------------------------------------------
# Step 3: Extract and Interpret the Results
# ---------------------------------------------------------
print("\n[Regression Results Summary]")
print(model.summary().tables[1])

# Extract the specific causal impact coefficient
causal_impact = model.params['did_interaction']
p_value = model.pvalues['did_interaction']

print(f"\n[Conclusion]")
print(f"Causal Impact of New Strategy: {causal_impact * 100:.2f}%")
print(f"Statistical Significance (p-value): {p_value:.4f}")

if p_value < 0.05:
    if causal_impact > 0:
        print("Verdict: The new targeting strategy caused a statistically significant INCREASE in recovery rates.")
    else:
        print("Verdict: The new targeting strategy caused a statistically significant DECREASE in recovery rates.")
else:
    print("Verdict: The change in recovery rates is NOT statistically significant. The new strategy had no measurable causal effect.")

# ---------------------------------------------------------
# Step 4: Calculate the Counterfactual (Financial Impact)
# ---------------------------------------------------------
# What would recovery have been without the strategy change?
if p_value < 0.05 and causal_impact > 0:
    total_treated_principal_post = reg_df[(reg_df['is_treated']==1) & (reg_df['post_change']==1)]['outstanding_amount'].sum()
    incremental_recovery_inr = total_treated_principal_post * causal_impact
    
    print(f"\n[Financial Counterfactual]")
    print(f"If we had NOT changed the strategy, we would have recovered ₹{incremental_recovery_inr:,.2f} LESS.")
else:
    print(f"\n[Financial Counterfactual]")
    print("If we had NOT changed the strategy, recovery would have been exactly the same (or the difference is just statistical noise).")

Loading Golden Dataset for Causal Analysis...

🧪 COUNTERFACTUAL ANALYSIS: DIFFERENCE-IN-DIFFERENCES
Running OLS Regression...

[Regression Results Summary]
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               0.4060      0.013     30.816      0.000       0.380       0.432
is_treated         -0.0356      0.019     -1.916      0.055      -0.072       0.001
post_change        -0.0749      0.026     -2.910      0.004      -0.125      -0.024
did_interaction     0.1314      0.036      3.609      0.000       0.060       0.203

[Conclusion]
Causal Impact of New Strategy: 13.14%
Statistical Significance (p-value): 0.0003
Verdict: The new targeting strategy caused a statistically significant INCREASE in recovery rates.

[Financial Counterfactual]
If we had NOT changed the strategy, we would have recovered ₹464,746,241.99 LESS.


### Methodology: Difference-in-Differences (DiD)

**Identification Strategy**
The Difference-in-Differences (DiD) approach isolates the causal impact of the new targeting strategy. It calculates the change in recovery rates before and after the strategy shift for the affected accounts (Treatment), and subtracts the change in recovery rates for a similar baseline group over the exact same time period (Control). This effectively subtracts natural time-based trends (like seasonality or economic shifts) to reveal the true incremental value of the new strategy.

**Treatment Group**
Accounts that were subjected to the **new** targeting strategy following the mid-year cutoff date (e.g., 'HIGH' and 'MEDIUM' risk segments).

**Control Group**
Accounts that continued to be worked under the **old** targeting strategy after the mid-year cutoff (e.g., 'LOW' and 'NPA' segments). *Note: If no explicit control group was held back, we will construct one from segments where the rollout was delayed, using Propensity Score Matching (PSM) to ensure baseline characteristics mirror the Treatment group.*

**Assumptions**

* **Parallel Trends (Crucial):** In the absence of the targeting change, the recovery rates of the Treatment and Control groups would have moved in parallel over time.
* **No Spillover (SUTVA):** Agents working the new targeting strategy did not accidentally apply those learnings or behaviors to the accounts in the Control group.
* **Stable Mix:** The underlying quality of debt assigned to both groups did not fundamentally change at the exact moment the strategy changed.

**Confounding Factors**

* **Agent Reallocation:** If the highest-performing agents were simultaneously reassigned to the new targeting strategy, the observed uplift would be driven by agent skill, not the strategy itself.
* **Seasonality and Macro-events:** Variations in borrower behavior (e.g., festive seasons, bonus payouts) that disproportionately impact the demographic makeup of the Treatment group compared to the Control group.
* **Concurrent Operational Changes:** If the telephony vendor or SMS frequency rules were updated at the exact same time as the targeting change, separating the effects becomes difficult.

**Limitations**

* **Global Rollout Constraint:** If the business executed a global rollout on a single day without leaving any accounts under the old strategy, the DiD methodology cannot be used. (In such a scenario, Bayesian Structural Time Series would be required to forecast a synthetic counterfactual, which is inherently less reliable).
* **Short Evaluation Window:** If the strategy changed recently, the maturity curve of the debt makes it difficult to estimate the ultimate lifetime recovery value reliably.

 ### 5 - Production Analytics Design

### Data Contracts

Agreements between data producers (e.g., telephony vendors, payment gateways) and data consumers (the analytics team) that enforce strict expectations on schema, data types, and allowed values. For this pipeline, a contract ensures `payment_status` only accepts 'SUCCESS', 'PENDING', or 'FAILED', and that `event_at` is always passed as a valid timestamp. If a vendor breaks the contract by sending a new status code, the pipeline halts or routes the data to a dead-letter queue rather than corrupting downstream metrics.

### Primary Keys

Unique identifiers assigned to each record to guarantee uniqueness and enable accurate table joins. In the collections dataset, `payment_id` serves as the primary key for the `payments` table, and `account_id` for the `accounts` table. Enforcing primary keys in the cleansing layer prevents duplicates (like the 500 duplicate payments identified in the forensics phase) from artificially inflating recovery rates.

### Metric Definitions

Centralized, codified formulas establishing exactly how business performance is calculated, preventing different teams from reporting conflicting numbers. In this system, "True Recovery Rate" is universally defined in the Golden Dataset as `SUM(total_recovered_inr) / SUM(outstanding_amount)` where `payment_status = 'SUCCESS'`.

### Data Lineage

The visual or programmatic map tracing the lifecycle of data from its raw source to its final dashboard visualization. For the 30-day cohort recovery metric, the lineage tracks back from the Dashboard → `obt_recovery_golden` (Serving) → `pd.merge_asof` attribution logic (Feature) → UTC normalized timestamps (Staging) → `calls.csv` and `payments.csv` (Raw). Lineage is critical for root-cause analysis when dashboard numbers look incorrect.

### Incremental Processing

A compute-efficient pipeline strategy where only new or updated records are processed during a job run, rather than recalculating the entire historical dataset. The system uses a "high-water mark" (e.g., `MAX(updated_at)`) from the previous run to only query and append yesterday's new payments and call attempts into the staging and golden layers.

### Late-Arriving Data

Data that enters the system after its expected processing window, often due to offline field agents syncing devices late or vendor API outages. The pipeline handles this using idempotent operations (like `MERGE` or `UPSERT` using primary keys) and a lookback window in the orchestration tool (e.g., Airflow processing `execution_date` minus 3 days) to safely ingest delayed events without duplicating existing records.

### Backfills

The intentional reprocessing of historical data to apply new business logic or correct systemic errors. If leadership decides to change the last-touch payment attribution window from 7 days to 14 days, a backfill job is triggered to drop the existing historical metrics and rebuild the entire Golden Dataset from the raw data using the updated 14-day rule.

### Data-Quality Checks

Automated assertions executed at each stage of the pipeline to validate data integrity before it reaches the dashboard. Examples include verifying `COUNT(payment_id) = COUNT(DISTINCT payment_id)` (no duplicates), checking that no call durations are negative, and ensuring `outstanding_amount` is not null. Tools like dbt or Great Expectations execute these checks daily.

### Monitoring

The operational oversight of the pipeline's physical health, execution state, and data freshness. Monitoring triggers alerts (via Slack or PagerDuty) if the daily ingestion job fails, if the database cluster runs out of memory, or if the `payments` table has not received new rows in over 24 hours.

### Anomaly Detection

Statistical monitoring applied to the *business metrics* rather than the pipeline infrastructure. If the overall collection rate drops suddenly from 12% to 2% overnight, or if the percentage of 'High Risk' accounts spikes uncharacteristically, anomaly detection algorithms (like Z-score thresholds or isolation forests) alert leadership to investigate a potential operational outage or denominator manipulation.